# Juego 2D en Pygame con MLP (paso a paso)

Este notebook explica el funcionamiento completo del proyecto `1-game/`:
un juego 2D en Pygame que registra las decisiones de un jugador humano para
entrenar un modelo de Machine Learning (MLP o Arbol de Decision) y luego jugar
de forma automatica imitando el estilo del jugador.

**Stack tecnologico**

- `pygame >= 2.5.0` para el juego y el render
- `scikit-learn >= 1.3.0` para `MLPClassifier` y `DecisionTreeClassifier`
- `numpy` y `matplotlib` para visualizacion

**Como ejecutar el juego**

```
cd 1-game/
python -m src.main
```

**Convenciones**

- Las referencias al codigo se muestran como `ruta/archivo.py:linea`.
- Los snippets son copia fiel del proyecto; no se modifica ninguna linea.
- El proyecto esta organizado por capas para que cada parte se pueda explicar
  de forma aislada.


## 1. Arquitectura por capas

El proyecto divide responsabilidades en cinco capas. Cada carpeta de `src/`
tiene una unica responsabilidad para que el flujo se pueda leer y explicar
por separado.

```
1-game/
|-- src/
|   |-- main.py            # punto de entrada
|   |-- core/              # constantes y tipos compartidos
|   |-- juego/             # loop, estado, fisica, bala, puntaje
|   |-- ml/                # dataset, entrenamiento, decision
|   |-- render/            # carga de assets y dibujado
|   '-- ui/                # menu y mensajes en pantalla
|-- assets/                # sprites, fondos, menu
|-- docs/                  # documentacion escrita (md)
|-- datos_mlp.csv          # dataset exportado
'-- requirements.txt
```

**Mapa rapido de archivos clave**

| Archivo | Una linea de responsabilidad |
|---|---|
| `src/main.py` | crea `Juego` y arranca el loop |
| `src/core/constantes.py` | tamanos base, escala, colores |
| `src/core/tipos.py` | `Sample` y dataclasses de estado |
| `src/juego/juego.py` | orquestador: input, IA, update, render |
| `src/juego/loop.py` | dispatcher de eventos de teclado |
| `src/juego/estado.py` | dataclasses de jugador, bala, score, modelo |
| `src/juego/fisica.py` | reglas puras de salto y agachado |
| `src/juego/bala.py` | disparo, movimiento y reset de la bala |
| `src/juego/puntaje.py` | bonus por esquiva |
| `src/ml/dataset.py` | registro y exportacion de muestras |
| `src/ml/modelo.py` | entrenamiento y decision del MLP |
| `src/ml/arbol.py` | entrenamiento y decision del Arbol |
| `src/ml/visualizacion.py` | graficos 2D y 3D del dataset |
| `src/render/activos.py` | carga y escala de assets |
| `src/render/dibujar.py` | funciones de blit en pantalla |
| `src/ui/menu.py` | menu principal con imagen de fondo |

Flujo general de un frame:

```
Input --> Estado --> IA --> Update --> Render
   ^                                |
   '--------------------------------'
```


## 2. Paso 1: constantes y tipos compartidos

Las constantes evitan numeros magicos repetidos por el codigo. Los tipos
compartidos permiten que las capas se comuniquen sin acoplarse a Pygame.

**Constantes** (`src/core/constantes.py`)

```python
BASE_W = 1080
BASE_H = 720
WINDOW_FRACTION = 0.97
EXTRA_SCALE = 1.1

BLANCO  = (255, 255, 255)
NEGRO   = (0, 0, 0)
GRIS    = (200, 200, 200)
AMARILLO = (255, 220, 120)
```

El juego arranca a `1080x720` y al pasar a fullscreen se reescala todo
multiplicando por `scale = min(w/BASE_W, h/BASE_H) * EXTRA_SCALE`.

**Tipos** (`src/core/tipos.py`)

El `Sample` es la unidad de dato que se guarda cada vez que el jugador toma
una decision. Es la pieza central del dataset.

```python
@dataclass
class Sample:
    velocidad_bala: float
    distancia: float
    salto: int
    bala_y: float
    bala_arriba: int
    agachado: int
    accion: int          # 0 nada | 1 salta | 2 agacha
    puntaje: int
    ataque_color: int
```

Tambien se definen `EstadoJugador`, `EstadoBala` y `EstadoModelo` como
dataclasses auxiliares. En el archivo `src/juego/estado.py` se usan dataclasses
mas completas (`PlayerState`, `BulletState`, `ModelState`, ...) que envuelven
rectangulos de Pygame y flags del juego.


## 3. Paso 2: carga de assets

Los assets viven en `assets/sprites/` y `assets/game/`. El modulo
`src/render/activos.py` se encarga de cargarlos, escalarlos al tamano actual
y devolver un fallback coloreado si el archivo falta (asi el juego no se
rompe si un PNG se borra por accidente).

```python
def cargar_superficie_segura(path, size, fallback_color=(200, 200, 200, 255)):
    try:
        img = pygame.image.load(path).convert_alpha()
        return pygame.transform.smoothscale(img, size)
    except Exception:
        surf = pygame.Surface(size, pygame.SRCALPHA)
        surf.fill(fallback_color)
        return surf
```

La funcion `cargar_activos()` devuelve un diccionario con todos los sprites
que la clase `Juego` necesita para renderizar.

**Inventario**

- Jugador: 4 frames de caminata + sprite de salto + sprite de agachado
- Bala: imagen roja (ataque alto) y amarilla (ataque bajo)
- Nave enemiga: `enemy.png`
- Fondo: `background.png` (con scroll horizontal)
- Menu: `title.png` cubre toda la pantalla

Las imagenes se reescalan cada vez que cambia la resolucion para mantener
proporciones (el factor `scale` aparece en casi todos los calculos).


## 4. Paso 3: fisica del jugador

La fisica esta aislada en `src/juego/fisica.py`. Son funciones puras que
reciben un `pygame.Rect` y devuelven flags: asi se pueden testear sin abrir
una ventana.

**Salto**

```python
def iniciar_salto(player, en_suelo, agachado):
    if en_suelo and not agachado:
        return True
    return False


def aplicar_salto(player, salto, salto_vel, gravedad, ground_y):
    if salto:
        player.y -= int(salto_vel)
        salto_vel -= gravedad
        if player.y >= ground_y:
            player.y = ground_y
            salto = False
            salto_vel = 0.0
            en_suelo = True
        else:
            en_suelo = False
        return salto, salto_vel, en_suelo
    return salto, salto_vel, True
```

La velocidad inicial (`salto_vel_inicial`) se escala con la resolucion y se
acerca a un `1.15x` para que el salto se sienta natural. La gravedad hace el
resto del trabajo. La forma resultante es la tipica parabola:

```
y
|^  .       .       .
| '.   .       .       .
|   '.   .       .       .
|     '.   .       .       .
|       '.   .       .       .
|         '.   .       .       .
+-----------'----'-------'------> frames
           min          aterrizaje
```

**Agachado**

```python
def iniciar_agacharse(player, agachado, en_suelo, altura_base):
    if agachado or not en_suelo:
        return agachado
    nueva_altura = max(1, int(altura_base * 0.1))
    player.height = nueva_altura
    return True


def terminar_agacharse(player, agachado, altura_base):
    if not agachado:
        return False
    player.height = altura_base
    return False
```

Al agacharse la altura del rect del jugador baja al 10% de la base. Eso
permite esquivar balas que vienen bajas. Ademas, mientras esta agachado
no se puede iniciar un nuevo salto.


## 5. Paso 4: logica de la bala

La bala se modela en `src/juego/bala.py`. El ciclo de vida de una bala es:

```
[no disparada] --disparar_bala()--> [volando] --sale de pantalla--> [no disparada]
                                          |
                                          +-- colision --> reset_juego()
```

**Disparo**

```python
def disparar_bala(bala, disparada, velocidad_base, scale, bala_arriba,
                  ground_y, player_height, bullet_height):
    if disparada:
        return True, velocidad_base, bala_arriba
    velocidad = int(random.randint(-12, -6) * scale)
    bala_arriba = random.choice([True, False])
    if bala_arriba:
        bala.y = ground_y + player_height - int(bullet_height)
    else:
        bala.y = ground_y + int(player_height * 0.35) - int(30 * scale)
    return True, velocidad, bala_arriba
```

Cada bala tiene velocidad horizontal aleatoria en `[-12, -6] * scale` y elige
entre dos alturas (alta o baja). El `arriba` se usa para decidir el sprite
(rojo vs amarillo) y se exporta al dataset.

**Movimiento y reset**

```python
def mover_bala(bala, velocidad, jugador_x, dist_min):
    bala.x += velocidad
    distancia = abs(jugador_x - bala.x)
    if dist_min is None:
        return distancia
    return min(dist_min, distancia)


def reset_bala(bala, start_x):
    bala.x = start_x
    return False, False, None
```

Ademas de mover la bala, `mover_bala()` registra la `dist_min` (la menor
distancia que tuvo con el jugador). Cuando la bala sale de pantalla por la
izquierda, esa distancia se usa para calcular el **bonus por esquiva**.


## 6. Paso 5: estado del juego

`src/juego/estado.py` define las dataclasses que modelan el estado completo
del juego. Tener el estado separado de la logica permite:

- Testear reglas pasando estados construidos a mano.
- Dibujar con la informacion ya estructurada.
- Cambiar el `RenderState` o el `ModelState` sin tocar la fisica.

```python
@dataclass
class PlayerState:
    rect: pygame.Rect
    salto: bool
    en_suelo: bool
    salto_vel: float
    agachado: bool


@dataclass
class BulletState:
    rect: pygame.Rect
    velocidad: int
    disparada: bool
    arriba: bool
    dist_min: Optional[int]


@dataclass
class ScoreState:
    valor: int
    por_frame: int
    esquiva_base: int
    esquiva_umbral: int


@dataclass
class ModelState:
    datos: List[Sample]
    modelo_mlp: Optional[object]
    scaler: Optional[object]
    entrenado_mlp: bool
    clase_unica_mlp: Optional[int]
    modelo_arbol: Optional[object]
    entrenado_arbol: bool
    clase_unica_arbol: Optional[int]
    ultima_proba: Optional[float]
    accion_auto: str
```

`ModelState` concentra todo lo relacionado con el modelo en memoria: el
dataset (`datos`), los modelos entrenados (MLP y Arbol), sus scalers, las
metricas auxiliares y la ultima accion que la IA eligio.


## 7. Paso 6: el loop principal

El corazon del juego es la clase `Juego` en `src/juego/juego.py`. Su metodo
`loop()` corre un reloj a 45 FPS y, en cada iteracion, ejecuta cinco fases:

1. **Procesar eventos** (teclado, cierre de ventana).
2. **Decidir la accion del jugador** (input manual o prediccion de la IA).
3. **Disparar la bala** si no hay una activa.
4. **Sumar puntaje** y actualizar el frame (`_update_frame`).
5. **Renderizar** y flipear la pantalla.

**Dispatcher de eventos** (`src/juego/loop.py`)

```python
def procesar_eventos(eventos, modo_auto, en_suelo, on_quit, on_menu,
                     on_fullscreen, on_jump, on_crouch_start, on_crouch_end):
    for e in eventos:
        if e.type == pygame.QUIT:
            on_quit()
        elif e.type == pygame.KEYDOWN:
            if e.key == pygame.K_q:
                on_quit()
            elif e.key in (pygame.K_ESCAPE, pygame.K_p):
                on_menu()
            elif e.key == pygame.K_f:
                on_fullscreen()
            elif e.key == pygame.K_SPACE and (not modo_auto) and en_suelo:
                on_jump()
            elif e.key in (pygame.K_DOWN, pygame.K_s):
                if not modo_auto:
                    on_crouch_start()
        elif e.type == pygame.KEYUP:
            if e.key in (pygame.K_DOWN, pygame.K_s):
                if not modo_auto:
                    on_crouch_end()
```

El dispatcher recibe callbacks. Eso permite que `Juego` decida que hacer en
cada caso (saltar, agacharse, ir al menu, etc.) sin que `loop.py` conozca
los detalles del juego. El modo automatico ignora las teclas de juego.

**Loop de `Juego`** (esqueleto, `src/juego/juego.py:628`)

```python
def loop(self):
    reloj = pygame.time.Clock()
    self.mostrar_menu()

    while self.corriendo:
        procesar_eventos(pygame.event.get(), self.modo_auto,
                          self.player.en_suelo, ...)

        if not self.corriendo:
            break

        if self.modo_auto:
            accion, _ = self._decidir_accion_auto()
            # aplicar accion al PlayerState
        else:
            self._actualizar_patron_agacharse_manual()
            self.registrar_decision_manual()

        if not self.bullet.disparada:
            self.bullet.disparada, ... = disparar_bala(...)

        self.score.valor += self.score.por_frame
        self._update_frame()
        pygame.display.flip()
        reloj.tick(45)
```

**Diagrama de flujo por frame**

```
+-------------------+
| procesar eventos  |
+---------+---------+
          |
          v
+-------------------+    no
| modo_auto activo? +----------+
+---------+---------+          |
          | si                 |
          v                    v
+-------------------+   +--------------------+
| decidir con IA    |   | registrar decision |
+---------+---------+   |     (manual)       |
          |             +---------+----------+
          |                       |
          +-----------+-----------+
                      |
                      v
+-------------------+
| bala ya disparada?|
+---+-------+-------+
    | no    | si
    v       |
+--------+  |
|disparar|  |
+---+----+  |
    |       |
    +---+---+
        |
        v
+-------------------+
| puntaje y update  |
+---------+---------+
          |
          v
+-------------------+
| render + flip     |
+-------------------+
```


## 8. Paso 7: menu principal

`src/ui/menu.py` arma la pantalla inicial. El fondo es la imagen
`assets/game/title.png` escalada a pantalla completa con una vineta oscura
para que las opciones se lean bien. Encima se dibuja un panel con:

- Una lista de teclas (`M`, `A`, `T`, `R`, `D`, `C`, `F`, `Q`) y su texto.
- Una linea de estado con la cantidad de muestras, si hay MLP o Arbol
  entrenado, y la resolucion actual.
- Un mensaje contextual cuando se entrena o se exporta un CSV.

Las opciones que se ofrecen son:

| Tecla | Accion |
|---|---|
| `M` | modo manual: reinicia dataset y borra el modelo |
| `A` | modo automatico con MLP (si esta entrenado) |
| `T` | entrenar MLP |
| `R` | modo automatico con Arbol de Decision |
| `D` | entrenar Arbol de Decision |
| `C` | exportar dataset a `datos_mlp.csv` |
| `F` | alternar fullscreen |
| `Q` | salir |

Teclas del juego (solo en modo manual):

| Tecla | Accion |
|---|---|
| `ESPACIO` | saltar |
| `ABAJO` o `S` | agacharse (mantener presionado) |
| `ESC` o `P` | volver al menu |

Si se intenta entrar al modo automatico sin haber entrenado antes, el menu
muestra un mensaje en color amarillo explicando que primero hay que entrenar.


## 9. Paso 8: modo manual y registro del dataset

El modo manual es donde se construye el dataset. Cada vez que hay una bala
volando, `Juego.registrar_decision_manual()` registra un `Sample` con la
intencion del jugador en ese instante.

**Decision clave del proyecto**: la etiqueta `accion` se toma de la **entrada
del usuario**, no del estado fisico del personaje. Asi, si el jugador se
agacha todo el tiempo aunque no haya bala, esa preferencia queda registrada
y la IA podra imitarla en modo automatico.

**Codigo de registro** (`src/juego/juego.py:290`)

```python
def registrar_decision_manual(self):
    accion = (
        1
        if self.player.salto or self._accion_manual == 1
        else 2
        if self.player.agachado or self._crouch_hold
        else 0
    )
    ataque_color = 1 if self.bullet.arriba else 0
    self._estilo_hist.append(accion)
    registrar_decision_manual(
        self.model.datos,
        self.bullet.disparada,
        self.jugador.x,
        self.bala.x,
        self.bala.y,
        self.player.en_suelo,
        self.player.agachado,
        accion,
        self.bullet.velocidad,
        self.score.valor,
        self.bullet.arriba,
        ataque_color,
    )
    self._accion_manual = 0
```

**Codigo de almacenamiento** (`src/ml/dataset.py:8`)

```python
def registrar_decision_manual(datos_modelo, bala_disparada, jugador_x,
                              bala_x, bala_y, en_suelo, agachado, accion,
                              velocidad_bala, puntaje, bala_arriba,
                              ataque_color):
    if not bala_disparada:
        return
    distancia = abs(jugador_x - bala_x)
    salto_label = 0 if en_suelo else 1
    datos_modelo.append(
        Sample(
            velocidad_bala=float(velocidad_bala),
            distancia=float(distancia),
            salto=salto_label,
            bala_y=float(bala_y),
            bala_arriba=1 if bala_arriba else 0,
            agachado=1 if agachado else 0,
            accion=int(accion),
            puntaje=int(puntaje),
            ataque_color=int(ataque_color),
        )
    )
```

**Exportacion a CSV**

La funcion `exportar_datos_csv(datos_modelo, base_dir)` escribe el dataset en
`datos_mlp.csv` con la cabecera:

```
velocidad_bala,distancia,salto,bala_y,bala_arriba,agachado,accion,puntaje,ataque_color
```

Cada fila es una decision del jugador. Por ejemplo:

```
-9.0,961.0,0,535.0,1,0,0,1,1
-9.0,952.0,0,535.0,1,0,0,2,1
-9.0,943.0,0,535.0,1,0,0,3,1
```

El archivo del proyecto ya trae ~520 muestras de ejemplo que se pueden usar
para entrenar sin necesidad de jugar manualmente.


In [ ]:
# Vista rapida del dataset ya generado
import csv
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "1-game":
    PROJECT_ROOT = PROJECT_ROOT / "1-game"
ruta = PROJECT_ROOT / "datos_mlp.csv"
with ruta.open() as f:
    filas = list(csv.reader(f))

print(f'Cantidad de filas (incluye cabecera): {len(filas)}')
print('Cabecera:', filas[0])
print('Primeras 3 muestras:')
for fila in filas[1:4]:
    print('  ', fila)

conteos = {0: 0, 1: 0, 2: 0}
for fila in filas[1:]:
    conteos[int(fila[6])] += 1
print('Distribucion de clases (0=nada, 1=salta, 2=agacha):', conteos)


## 10. Paso 9: entrenamiento del MLP

El MLP vive en `src/ml/modelo.py`. Entrena un `MLPClassifier` de scikit-learn
con tres neuronas en dos capas ocultas (`hidden_layer_sizes=(3, 3)`),
activacion ReLU y optimizador Adam.

**Caracteristicas (features) que ve el modelo**

- `velocidad_bala` (negativa, escala con la resolucion)
- `distancia` entre jugador y bala
- `bala_y` (altura de la bala)
- `bala_arriba` (1 si viene alta, 0 si viene baja)
- `puntaje` (puntaje acumulado en el momento de la decision)
- `ataque_color` (alias de `bala_arriba`, redundante pero util para analisis)

**Etiquetas**

- `0`: no hacer nada
- `1`: saltar
- `2`: agacharse

**Pipeline de entrenamiento**

```
muestras
   |
   v
+----------------------+
| al menos 80 muestras?|  no --> modelo trivial: siempre accion 0
+----------+-----------+
           | si
           v
+----------------------+
| al menos 2 clases?  |  no --> modelo trivial: siempre la unica clase
+----------+-----------+
           | si
           v
+----------------------+
| train_test_split     |  (80% train, 20% test, estratificado)
+----------+-----------+
           |
           v
+----------------------+
| StandardScaler       |  normaliza features
+----------+-----------+
           |
           v
+----------------------+
| MLPClassifier        |  (3, 3) ReLU, Adam, max_iter=300000
+----------+-----------+
           |
           v
+----------------------+
| score en test -> acc |
+----------------------+
```

**Codigo** (`src/ml/modelo.py:10`)

```python
def entrenar_modelo(datos_modelo):
    samples = list(datos_modelo)
    if len(samples) < 80:
        return None, None, 0,
            'Modelo trivial: SIEMPRE NADA. Junta datos (>= 80).'

    X = [
        [s.velocidad_bala, s.distancia, s.bala_y,
         float(s.bala_arriba), float(s.puntaje), float(s.ataque_color)]
        for s in samples
    ]
    y = [s.accion for s in samples]
    clases = sorted(set(y))
    if len(clases) < 2:
        return None, None, int(clases[0]),
            'Modelo trivial: SIEMPRE la unica clase.'

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    clf = MLPClassifier(
        hidden_layer_sizes=(3, 3),
        activation='relu',
        solver='adam',
        max_iter=300000,
        random_state=42,
    )
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    return clf, scaler, None, f'MLP entrenado. Accuracy test ~ {acc:.3f}'
```


## 11. Paso 10: Arbol de Decision

`src/ml/arbol.py` entrena un `DecisionTreeClassifier` con `max_depth=6` y
`min_samples_leaf=4`. El preprocesamiento es identico al del MLP: misma
extraccion de features, mismo split 80/20 estratificado, mismas reglas de
minimo de muestras y de caso trivial.

**Codigo** (`src/ml/arbol.py:9`)

```python
def entrenar_arbol(datos_modelo):
    samples = list(datos_modelo)
    if len(samples) < 80:
        return None, 0,
            'Modelo trivial: SIEMPRE NADA. Junta datos (>= 80).'

    X = [
        [s.velocidad_bala, s.distancia, s.bala_y,
         float(s.bala_arriba), float(s.puntaje), float(s.ataque_color)]
        for s in samples
    ]
    y = [s.accion for s in samples]
    clases = sorted(set(y))
    if len(clases) < 2:
        return None, int(clases[0]),
            'Modelo trivial: SIEMPRE la unica clase.'

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    clf = DecisionTreeClassifier(
        random_state=42, max_depth=6, min_samples_leaf=4
    )
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    return clf, None, f'Arbol entrenado. Accuracy test ~ {acc:.3f}'
```

**Comparacion rapida**

| Aspecto | MLP | Arbol |
|---|---|---|
| Estructura | 6 -> 3 -> 3 -> 3 | reglas if/else aprendidas |
| Preprocesamiento | `StandardScaler` obligatorio | no necesita escalado |
| Interpretabilidad | caja negra | se puede graficar el arbol |
| Costo de entrenamiento | mayor (backprop + Adam) | menor (particiones greedy) |
| Mejor para | generalizar a patrones no vistos | mostrar reglas explicitas |

Ambos modelos comparten el mismo `Sample` y las mismas funciones de decision,
asi que el resto del juego no se entera de cual se eligio.


## 12. Paso 11: decision automatica en juego

En modo automatico, cada frame `Juego._decidir_accion_auto()` arma un vector
con el estado actual y se lo pasa al modelo entrenado.

**Codigo** (`src/ml/modelo.py:70`)

```python
def decision_auto_saltar(modelo, scaler, clase_unica, bala_disparada,
                          en_suelo, jugador_x, bala_x, bala_y,
                          velocidad_bala, puntaje, bala_arriba):
    if (not bala_disparada) or (not en_suelo):
        return 0, None

    distancia = abs(jugador_x - bala_x)

    if clase_unica is not None and modelo is None:
        proba_salto = 1.0 if clase_unica == 1 else 0.0
        return clase_unica, proba_salto

    if modelo is None or scaler is None:
        return 0, None

    X = [[
        float(velocidad_bala), float(distancia), float(bala_y),
        1.0 if bala_arriba else 0.0, float(puntaje),
        1.0 if bala_arriba else 0.0,
    ]]
    Xs = scaler.transform(X)
    if hasattr(modelo, 'predict_proba'):
        proba = modelo.predict_proba(Xs)[0]
        clases = list(modelo.classes_)
        proba_salto = None
        if 1 in clases:
            proba_salto = float(proba[clases.index(1)])
        pred = int(modelo.predict(Xs)[0])
        return pred, proba_salto
    pred = int(modelo.predict(Xs)[0])
    proba_salto = 1.0 if pred == 1 else 0.0
    return pred, proba_salto
```

**Reglas de seguridad**

- Si no hay bala o el jugador no esta en el suelo, la IA devuelve `0` (nada).
- Si el modelo no esta entrenado, devuelve `0` y `proba_salto=None`.
- Si solo hay una clase, devuelve esa clase como accion fija.

**Sesgo de estilo (imitar al jugador)**

`Juego` mantiene un historial de las ultimas 200 acciones manuales
(`_estilo_hist`). Si una accion supera el 55% del historial y hay al menos
40 muestras, la IA la prioriza aunque el modelo prediga otra cosa.

Esto hace que la IA **copie el estilo del jugador** (saltar siempre,
agacharse siempre, etc.) aunque no sea optimo. Si la accion dominante es
`2` (agachar), ademas se activa un ciclo de agachado/levantado para que la
postura se mantenga de forma natural.

**Visualizacion en pantalla**

Cuando hay un modelo entrenado y estamos en modo automatico, el juego
muestra la `proba_salto` y la `accion_auto` en la esquina superior izquierda,
asi uno puede ver que esta decidiendo la IA en tiempo real.


## 13. Paso 12: puntaje y bonus por esquiva

El puntaje se actualiza en `src/juego/puntaje.py`:

```python
def calcular_bonus_esquiva(dist_min, umbral, base):
    if dist_min is None:
        return 0
    umbral = max(1, umbral)
    bonus = max(0, umbral - int(dist_min))
    return base + bonus
```

Cada frame se suma `score.por_frame` al puntaje (1 punto por frame a 45 FPS).
Cuando la bala sale de la pantalla por la izquierda, se otorga un bonus:

- `base` (25 puntos) por haber esquivado la bala.
- `max(0, umbral - dist_min)` extra, proporcional a lo cerca que paso la
  bala del jugador. Mas cerca == mas bonus.

Si el jugador colisiona con la bala, se llama a `_reset_estado_juego()` que
reinicia las posiciones pero **no** resetea el dataset ni el modelo.


## 14. Paso 13: visualizacion del dataset

`src/ml/visualizacion.py` ofrece dos scatter para inspeccionar las muestras
recolectadas.

**Scatter 2D** (`graficar_datos_2d`): distancia jugador-bala vs velocidad de
la bala, en rojo las muestras en las que se salto y en azul las demas.

**Scatter 3D** (`graficar_datos_3d`): mismas variables, agregando el indice
de la muestra en el tiempo para ver la secuencia.

Si se quiere ejecutar la celda siguiente y se tiene matplotlib instalado,
se pueden regenerar las figuras desde el dataset del proyecto.


In [ ]:
import csv
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # no abre ventana; solo guarda el archivo
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "1-game":
    PROJECT_ROOT = PROJECT_ROOT / "1-game"
ruta = PROJECT_ROOT / "datos_mlp.csv"
with ruta.open() as f:
    reader = csv.DictReader(f)
    filas = list(reader)

xs = [float(f_['distancia']) for f_ in filas]
ys = [float(f_['velocidad_bala']) for f_ in filas]
cs = ['red' if int(f_['salto']) == 1 else 'blue' for f_ in filas]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(xs, ys, c=cs, alpha=0.6, edgecolors='k', s=30)
ax.set_xlabel('Distancia jugador-bala')
ax.set_ylabel('Velocidad bala')
ax.set_title('Datos entrenamiento MLP (rojo=salto, azul=no salto)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
salida = Path('/tmp/opencode/dataset_2d.png')
fig.savefig(salida, dpi=120)
plt.close(fig)
print(f'Figura 2D guardada en {salida}')

zs = list(range(len(filas)))
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(xs, ys, zs, c=cs, alpha=0.6, edgecolors='k', s=30)
ax.set_xlabel('Distancia')
ax.set_ylabel('Velocidad bala')
ax.set_zlabel('Indice (tiempo aproximado)')
ax.set_title('Datos entrenamiento MLP 3D (rojo=salto, azul=no salto)')
plt.tight_layout()
salida3d = Path('/tmp/opencode/dataset_3d.png')
fig.savefig(salida3d, dpi=120)
plt.close(fig)
print(f'Figura 3D guardada en {salida3d}')


## 15. Paso 14: flujo end-to-end

```
+------------+
| main.py    |
+-----+------+
      | crea
      v
+------------+
| Juego()    |
+-----+------+
      | loop()
      v
+-------------------+
| mostrar_menu()    |  (M, A, T, R, D, C, F, Q)
+-----+-------------+
      | elige modo
      v
+-------------------+
| modo manual (M)   |   --> registra decisiones en `datos`
+-------------------+
+-------------------+
| entrenar (T / D)  |   --> MLP o Arbol de Decision
+-------------------+
+-------------------+
| modo auto (A / R) |   --> la IA decide saltar / agachar
+-------------------+

Durante el loop, en cada frame:

1. **Eventos** -> callbacks de salto, agachado, menu, fullscreen, salida.
2. **Decision** -> manual (registra Sample) o auto (modelo + sesgo de estilo).
3. **Disparo** -> si no hay bala activa, se genera una con altura y velocidad
   aleatorias.
4. **Update** -> fisica del jugador, movimiento de la bala, colisiones,
   puntaje.
5. **Render** -> fondo con scroll, jugador (anim, jump, down), nave, bala,
   textos del modelo y del puntaje.


## 16. Resumen y ejecucion

**Ejecucion del juego**

Desde la carpeta `1-game/`:

```
pip install -r requirements.txt
python -m src.main
```

**Flujo recomendado**

1. Jugar un rato en modo manual (`M`) para generar >= 80 muestras.
2. Entrenar el MLP con `T` (y/o el Arbol con `D`).
3. Pasar a modo automatico con `A` (MLP) o `R` (Arbol) y observar como la
   IA intenta imitar el estilo registrado.
4. Exportar el dataset a CSV con `C` si se quiere analizar afuera.

**Limitaciones conocidas**

- Con menos de 80 muestras el modelo es trivial (siempre accion 0).
- El sesgo de estilo prioriza la moda aunque sea un estilo de juego malo.
- La bala se elige de forma aleatoria: no hay una dificultad progresiva.

**Posibles mejoras**

- Persistir el dataset y los modelos entrenados entre sesiones.
- Agregar mas features (p. ej. tendencia de la distancia, frames desde la
  ultima esquiva).
- Comparar MLP y Arbol lado a lado con metricas de desempeno en el juego.
- Implementar un sistema de jefes con vida escalada (hay una propuesta en
  `docs/enemigos.md`).

**Archivos del proyecto**

- Codigo fuente: `1-game/src/`
- Assets: `1-game/assets/`
- Documentacion escrita: `1-game/docs/`
- Dataset exportado: `1-game/datos_mlp.csv`
- Dependencias: `1-game/requirements.txt`
